In [39]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [40]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)

pip_install("mne", "numpy", "pandas", "scikit-learn", "tensorflow", "awscli")

print("Dependencies installed")

Dependencies installed


In [41]:
import os
from pathlib import Path

if os.path.exists('/content'):
    BASE = Path("/content/eeg-debug")
elif os.path.exists('/kaggle/working'):
    BASE = Path("/kaggle/working/eeg-debug")
else:
    BASE = Path.cwd() / "eeg-debug"

DATA_DIR = BASE / "data"
DS_DIR = DATA_DIR / "ds004504"
DERIV_DIR = DS_DIR / "derivatives"
PARTICIPANTS = DS_DIR / "participants.tsv"

for d in [DATA_DIR, DS_DIR, DERIV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Base:", BASE)

Base: /content/eeg-debug


In [42]:
import subprocess
import pandas as pd

def download_few_subjects(n=12):
    print("Downloading dataset (small subset)...")

    subprocess.run([
        "aws", "s3", "cp",
        "s3://openneuro.org/ds004504/participants.tsv",
        str(PARTICIPANTS),
        "--no-sign-request"
    ], check=True)

    df = pd.read_csv(PARTICIPANTS, sep="\t")
    subjects = df["participant_id"].head(n).tolist()

    for sid in subjects:
        print("Downloading:", sid)
        subprocess.run([
            "aws", "s3", "sync",
            f"s3://openneuro.org/ds004504/derivatives/{sid}/",
            str(DERIV_DIR / sid),
            "--no-sign-request"
        ], check=True)

    print("Done.")

download_few_subjects(12)

download: s3://openneuro.org/ds004504/participants.tsv to ../../content/eeg-debug/data/ds004504/participants.tsv
Downloading: sub-001
Downloading: sub-002
Downloading: sub-003
Downloading: sub-004
Downloading: sub-005
Downloading: sub-006
Downloading: sub-007
Downloading: sub-008
Downloading: sub-009
Downloading: sub-010
Downloading: sub-011
Downloading: sub-012
Done.


In [43]:
import pandas as pd

GROUP_MAP = {'A': 0, 'F': 1, 'C': 2}

participants = pd.read_csv(PARTICIPANTS, sep="\t")
participants["label"] = participants["Group"].map(GROUP_MAP)
participants = participants.dropna(subset=["label"])

print(participants["label"].value_counts())

label
0    36
2    29
1    23
Name: count, dtype: int64


In [44]:
MAX_PER_CLASS = 8

participants_small = (
    participants.groupby("label", group_keys=False)
    .head(MAX_PER_CLASS)
    .reset_index(drop=True)
)

print("Subjects per class:")
print(participants_small["label"].value_counts())

Subjects per class:
label
0    8
2    8
1    8
Name: count, dtype: int64


In [45]:
import mne
import numpy as np

mne.set_log_level("ERROR")

def load_subject(path):
    raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
    raw.filter(0.5, 45, verbose=False)
    raw.notch_filter(50, verbose=False)
    return raw.get_data(), raw.info["sfreq"]

In [46]:
def make_epochs(data, sfreq, duration=2.0):
    step = int(duration * sfreq)
    n_ch, n_t = data.shape
    
    epochs = []
    for i in range(0, n_t - step, step):
        epochs.append(data[:, i:i+step])
    
    return np.array(epochs)

In [47]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

subjects = participants_small["participant_id"].values
labels   = participants_small["label"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)

train_idx, val_idx = next(gss.split(X=np.zeros(len(subjects)), y=labels, groups=subjects))

train_subj = participants_small.iloc[train_idx]["participant_id"].tolist()
val_subj   = participants_small.iloc[val_idx]["participant_id"].tolist()

print("Train subjects:", train_subj)
print("Val subjects:", val_subj)

Train subjects: ['sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-039', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-067', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073']
Val subjects: ['sub-001', 'sub-037', 'sub-038', 'sub-040', 'sub-066', 'sub-068']


In [48]:
X_train, y_train = [], []
X_val, y_val = [], []

for _, row in participants_small.iterrows():
    sid   = row["participant_id"]
    label = int(row["label"])

    fpath = DERIV_DIR / sid / "eeg" / f"{sid}_task-eyesclosed_eeg.set"

    if not fpath.exists():
        continue

    data, sfreq = load_subject(str(fpath))
    epochs = make_epochs(data, sfreq)

    if sid in train_subj:
        X_train.append(epochs)
        y_train.extend([label] * len(epochs))
    else:
        X_val.append(epochs)
        y_val.extend([label] * len(epochs))

X_train = np.concatenate(X_train)
X_val   = np.concatenate(X_val)

y_train = np.array(y_train)
y_val   = np.array(y_val)

print("Train:", X_train.shape)
print("Val:", X_val.shape)

/tmp/ipykernel_55/1895391049.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
/tmp/ipykernel_55/1895391049.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
/tmp/ipykernel_55/1895391049.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
/tmp/ipykernel_55/1895391049.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(path, preload=True, verbose=False)
/tmp/ipykernel_5

Train: (6544, 19, 1000)
Val: (2174, 19, 1000)


In [49]:
mean = X_train.mean()
std  = X_train.std() + 1e-8

X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std

X_train = X_train[..., np.newaxis]
X_val   = X_val[..., np.newaxis]

print("Final shapes:", X_train.shape, X_val.shape)

Final shapes: (6544, 19, 1000, 1) (2174, 19, 1000, 1)


In [50]:
import numpy as np

print("Train class distribution:", np.bincount(y_train))
print("Val class distribution:", np.bincount(y_val))

Train class distribution: [2399 1942 2203]
Val class distribution: [ 299  560 1315]


In [51]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_eegnet(n_channels, n_samples, n_classes):
    model = models.Sequential([
        layers.Conv2D(8, (1, 64), padding='same', input_shape=(n_channels, n_samples, 1)),
        layers.BatchNormalization(),

        layers.DepthwiseConv2D((n_channels, 1), depth_multiplier=2),
        layers.BatchNormalization(),
        layers.Activation('elu'),
        layers.AveragePooling2D((1, 4)),
        layers.Dropout(0.3),

        layers.SeparableConv2D(16, (1, 16), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('elu'),
        layers.AveragePooling2D((1, 4)),
        layers.Dropout(0.5),

        layers.Flatten(),
        layers.Dense(3, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_eegnet(X_train.shape[1], X_train.shape[2], 3)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 19, 1000, 8)    │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 19, 1000, 8)    │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d_3              │ (None, 1, 1000, 16)    │           320 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 1, 1000, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 1, 1000, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_6             │ (None, 1, 250, 16)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_3              │ (None, 1, 250, 16)     │           528 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 1, 250, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 1, 250, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_7             │ (None, 1, 62, 16)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1, 62, 16)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 992)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │         2,979 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,507 (17.61 KB)

 Trainable params: 4,427 (17.29 KB)

 Non-trainable params: 80 (320.00 B)

In [52]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

In [53]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 16s 47ms/step - accuracy: 0.3596 - loss: 1.3292 - val_accuracy: 0.1380 - val_loss: 1.2134
Epoch 2/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.5467 - loss: 0.9314 - val_accuracy: 0.2617 - val_loss: 1.3135
Epoch 3/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.7082 - loss: 0.6443 - val_accuracy: 0.5879 - val_loss: 0.9661
Epoch 4/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.7843 - loss: 0.4991 - val_accuracy: 0.6615 - val_loss: 0.8761
Epoch 5/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8079 - loss: 0.4438 - val_accuracy: 0.6403 - val_loss: 1.0524
Epoch 6/15
205/205 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.8350 - loss: 0.3876 - val_accuracy: 0.6509 - val_loss: 1.0494


In [54]:
loss, acc = model.evaluate(X_val, y_val)
print(f"Validation Accuracy: {acc:.4f}")

68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6953 - loss: 0.8297
Validation Accuracy: 0.6615
